# 4. LLM Fine-Tuning (Standard VQ-VAE Data)

**Objective**: Train the Large Language Model (LLM) using the dataset generated by the standard Vector Quantized VAE (VQ-VAE) pipeline.
In this version of the project, we teach the LLM to utilize a traditional Euclidean discrete bottleneck. The model learns to predict a sequence of `<latent_N>` tokens—representing a compressed, discrete version of its internal reasoning—before decoding that information into a final natural language solution. This serves as our baseline to compare against the geometrically-informed "Oddity" approach.


## 4.1 Environment Setup and Repository Cloning

To ensure reproducibility, this section automates the setup of the working environment:
1. **Google Drive Integration:** Mounts your personal Drive to store persistent data (checkpoints and processed datasets).
2. **Project Structure:** Automatically creates a `DLAI` folder in your Drive.
3. **Dependency Management:** Installs the `uv` package manager and resolves all requirements defined in `pyproject.toml`.
4. **Source Code:** Clones the `llama` branch from our GitHub repository to provide access to the `src` module and configuration files.

**Note for Evaluators:** Please authorize the Google Drive mount when prompted to allow the notebook to save and retrieve project files.

In [ ]:
import os, sys

# 1. Mount Google Drive
# Evaluators will need to accept the pop-up to connect their Drive
from google.colab import drive
drive.mount('/content/drive')

# 2. Setup directories on Drive
# Create the DLAI folder if it doesn't exist on their Drive
DRIVE_PROJECT_PATH = "/content/drive/MyDrive/DLAI"
if not os.path.exists(DRIVE_PROJECT_PATH):
    os.makedirs(DRIVE_PROJECT_PATH, exist_ok=True)
    print(f"Created project folder at: {DRIVE_PROJECT_PATH}")

# 3. UV Installation
# We use UV for much faster dependency management than standard pip
!curl -LsSf https://astral.sh/uv/install.sh | sh
os.environ['PATH'] = f"{os.path.expanduser('~')}/.cargo/bin:" + os.environ['PATH']

# 4. Clone the Repository (Branch: llama)
# If the local folder doesn't exist, clone the specific branch
%cd /content
if not os.path.exists("DLAI"):
    !git clone --branch llama https://github.com/irene-30/DLAI.git
else:
    print("Repo already exists, pulling latest changes...")
    !git -C DLAI pull

# 5. Synchronize pyproject.toml
# Copy the pyproject.toml from the cloned repo to the Drive folder (if necessary)
# or vice versa, to ensure that UV reads the correct dependencies.
!cp /content/DLAI/pyproject.toml {DRIVE_PROJECT_PATH}/pyproject.toml

# 6. Install dependencies via pyproject.toml
# This command reads the .toml file and installs everything necessary
%cd /content/DLAI
!uv pip install -e . --system

# 7. Add to the system path to allow imports from 'src'
sys.path.append("/content/DLAI")
%cd /content

print("✅ Setup completed successfully!")

In [ ]:
# --- Config Update for Latent Oddity ---
NUM_TRAIN_EPOCHS = 1
PER_DEVICE_TRAIN_BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 8
LEARNING_RATE = 2e-5
NUM_WORKERS = 2


DRIVE_SAVE_DIR = "/content/drive/My Drive/DLAI/experiments/vqvae_standard/llama_finetuning_results"


os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)
FINAL_MODEL_DIR = os.path.join(DRIVE_SAVE_DIR, "final_model")

### Step 1: Load and Pre-Tokenize the Standard Dataset
We import the JSONL file containing the training samples where mathematical text is interleaved with standard VQ-VAE discrete tokens. Pre-tokenizing the entire dataset before starting the training loop is a critical optimization: it ensures that the GPU remains fully utilized by preventing the CPU from becoming a bottleneck during data loading. We set up the labels to match the input IDs, enabling the model to learn the joint probability of both the latent reasoning sequence and the final answer.

In [ ]:
# 1. Load & Pre-Tokenize Oddity Data

import torch
from datasets import load_dataset
from torch.utils.data import DataLoader
from torch.optim import AdamW
from transformers import get_scheduler
from tqdm.auto import tqdm

from src.utils import get_llm_tokenizer, get_llm_model, LLM_MODEL_NAME

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

PATH_PROCESSED_DATA_ODDITY = "/content/drive/My Drive/DLAI/data/processed/metamath_assorted_standard.jsonl"

tokenizer = get_llm_tokenizer()

try:
    print(f"Loading Oddity dataset from: {PATH_PROCESSED_DATA_ODDITY}")
    raw_dataset = load_dataset("json", data_files=PATH_PROCESSED_DATA_ODDITY, split="train")
except FileNotFoundError:
    print(f"❌ Error: Can't find the file in {PATH_PROCESSED_DATA_ODDITY}. Make sure you finished Notebook 3!")

def tokenize_function(examples):
    tokenized = tokenizer(
        examples["text"],
        max_length=256,
        padding="max_length",
        truncation=True,
    )
    # Copy input_ids into labels for Causal Language Modeling (predicting the next token)
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

tokenized_dataset = raw_dataset.map(
    tokenize_function,
    batched=True,
    num_proc=os.cpu_count(),
    remove_columns=["text"]
)
tokenized_dataset.set_format("torch")

train_dataloader = DataLoader(
    tokenized_dataset,
    batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

### Step 2: Load Base Model and Setup LoRA
We initialize the base Llama-3 architecture and perform an embedding resize to accommodate the new `<latent_N>` tokens added to the vocabulary. To make the training feasible on hardware with limited VRAM (like the T4), we employ Low-Rank Adaptation (LoRA). This technique restricts updates to a small set of auxiliary parameters while keeping the backbone of the LLM frozen, allowing us to teach the model how to manipulate the new discrete tokens without losing its pre-trained linguistic and logical capabilities.

In [ ]:
# 2. Setup Model with potential vocabulary resize
#LLM_MODEL_NAME = "meta-llama/Llama-3.2-3B-Instruct"
model = get_llm_model(LLM_MODEL_NAME, len(tokenizer)).to(device)

# If you added new Riemannian tokens to the tokenizer, resize the model embeddings to match
if model.get_input_embeddings().weight.shape[0] != len(tokenizer):
    print(f"Resizing embeddings from {model.get_input_embeddings().weight.shape[0]} to {len(tokenizer)}")
    model.resize_token_embeddings(len(tokenizer))

optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)
num_training_steps = (NUM_TRAIN_EPOCHS * len(train_dataloader)) // GRADIENT_ACCUMULATION_STEPS
lr_scheduler = get_scheduler("linear", optimizer=optimizer, num_warmup_steps=0, num_training_steps=num_training_steps)

### Step 3: Execute Training with Hugging Face Trainer
The training process is managed through the `SFTTrainer` with a specific focus on memory efficiency. We use a combination of 8-bit paged optimizers and gradient accumulation to simulate a larger batch size while staying within the 16GB VRAM limit.

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer
from peft import LoraConfig
from transformers.trainer_utils import get_last_checkpoint

# 1. Critical configuration for T4 (16GB VRAM)
training_args = TrainingArguments(
    output_dir=DRIVE_SAVE_DIR,
    per_device_train_batch_size=1,       # Micro-batch
    gradient_accumulation_steps=16,      # Matches your "Effective Batch Size" of 16
    learning_rate=2e-4,                  # Standard for LoRA
    num_train_epochs=1,
    fp16=True if device=="cuda" else False, # Mixed precision (T4 doesn't support bf16)
    gradient_checkpointing=True,         # Massive memory saver
    optim="paged_adamw_8bit",            # Moves optimizer states to CPU/Quantizes them
    max_grad_norm=0.3,                   # Stability for QLoRA
    warmup_steps=0.03,
    lr_scheduler_type="constant",
    save_strategy="steps",  # Save memory by not writing checkpoints mid-run
    save_steps = 10,
    save_total_limit = 2,
    logging_steps = 10,
)


# 2. LoRA Settings (Essential for fitting on T4)
peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"], # Minimal targets to save VRAM
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# 3. Setup Trainer
trainer = SFTTrainer(
    model=model,
    train_dataset=tokenized_dataset,
    peft_config=peft_config,
    args=training_args,
)

# Crucial: Disable cache for training
model.config.use_cache = False

print("--- Starting LLM Fine-Tuning---")

# 1. Check if a valid checkpoint folder exists in your Drive directory
last_checkpoint = get_last_checkpoint(DRIVE_SAVE_DIR)

if last_checkpoint is not None:
    print(f"--- 🔄 Resuming from checkpoint: {last_checkpoint} ---")
else:
    print("--- 🆕 Starting fresh training (no checkpoint found) ---")

# 2. Pass the result (either a path or None) to the train method
trainer.train(resume_from_checkpoint=last_checkpoint)

# 1. Define the final destination
FINAL_OUTPUT_DIR = "/content/drive/My Drive/DLAI/experiments/vqvae_standard/llama_finetuning_results/final_model"
# 2. Save the model and tokenizer
# This saves the LoRA adapters and the updated config
trainer.save_model(FINAL_OUTPUT_DIR)
tokenizer.save_pretrained(FINAL_OUTPUT_DIR)

print(f"✅ Training complete! Final model saved to: {FINAL_OUTPUT_DIR}")